# Positive-class weighting for Funding and Other

Last focused classifier screen before a future Kaggle comparison. Start from the unchanged current DBSCAN + logistic regression model (sample F1 0.910256, macro F1 0.811421). Cross positive-example multipliers **1,1.5,2,3** for Funding and Other: **16 configurations**, including the original 1/1 control. These labels have only 8 and 25 verified positive training trials respectively; neither currently receives DBSCAN-generated labels.

Multiply the original raw fitting weight of positive Funding/Other examples by the selected factor, leave negative examples unchanged, then renormalize each reason's fitting weights to total **157**. This intentionally changes the positive/negative fitting influence without changing labels or source raw weights. Unknown targets remain excluded. Keep the other three classifiers exactly unchanged, the same frozen scaler, all 150 embeddings, C=1000, cutoff 0.5, and highest-probability fallback. Source generated targets/weights remain fixed.

Report sample, macro, per-reason F1 and Funding/Other precision, recall, false positives and false negatives. Fit only on the original training data; no validation-label refit, feature changes, pseudo-label regeneration, Kaggle submission, commit, or tag. Save exact effective weights, shared fixed data, fitted models, and predictions. Prepare a local candidate only if primary sample F1 improves; ties favor the original/lower-weight setting.

In [1]:
from pathlib import Path
import hashlib
import json
import platform
import warnings
import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import f1_score
from threadpoolctl import threadpool_limits
from IPython.display import display

REASONS=["Safety","Funding","Efficacy","Recruitment","Other"]
FEATURES=[f"emb_{i}" for i in range(150)]
POSITIVE_MULTIPLIERS=[1.0,1.5,2.0,3.0]
TARGET_REASONS=["Funding","Other"]
cwd=Path.cwd()
DATA=next(p for p in [cwd,cwd/"part1",cwd/"AML-Midterm"/"part1"]
          if (p/"part1_train.csv").is_file() or (p/"data/part1_train.csv").is_file()).resolve()
INPUT_DIR=DATA if (DATA/"part1_train.csv").is_file() else DATA/"data"
SOURCE=DATA/"experiments/additional_methods"
OUT=DATA/"experiments/rare_label_weights"
for folder in ["datasets","models","validation_predictions","submissions"]:(OUT/folder).mkdir(parents=True,exist_ok=True)
train=pd.read_csv(INPUT_DIR/"part1_train.csv");valid=pd.read_csv(INPUT_DIR/"part1_val.csv")
known=train[REASONS].isin([0,1]).all(axis=1).to_numpy()
original=train[REASONS].to_numpy(dtype=int);truth=valid[REASONS].to_numpy(dtype=int)
assert len(train)==1171 and known.sum()==157 and len(valid)==78
assert train.ID.is_unique and valid.ID.is_unique and set(train.ID).isdisjoint(valid.ID)
assert np.isin(truth,[0,1]).all()
record=json.loads((SOURCE/"submission_manifest.json").read_text())
source_model_path=SOURCE/record["model_file"];source_dataset_path=SOURCE/"datasets/dbscan_m5_q25.csv"
source_model=joblib.load(source_model_path)
fixed=pd.read_csv(source_dataset_path,float_precision="round_trip")
assert fixed.ID.tolist()==train.ID.tolist()
Y=fixed[REASONS].to_numpy(dtype=int);W=fixed[[f"weight_{r}" for r in REASONS]].to_numpy()
np.testing.assert_array_equal(Y[known],original[known]);np.testing.assert_array_equal(W[known],np.ones((157,5)))
assert np.array_equal(W>0,Y!=-100)
for reason in TARGET_REASONS:assert (W[~known,REASONS.index(reason)]==0).all()
(OUT/"datasets/fixed_source.csv").write_bytes(source_dataset_path.read_bytes())
scaler=source_model["scaler"];assert scaler.n_samples_seen_==157
XS=scaler.transform(train[FEATURES].to_numpy());VS=scaler.transform(valid[FEATURES].to_numpy())
protected_paths=[INPUT_DIR/"part1_train.csv",INPUT_DIR/"part1_val.csv",source_model_path,source_dataset_path,
                 SOURCE/record["submission_file"],SOURCE/"labeling_models/dbscan_m5_q25.joblib",SOURCE/"submission_manifest.json"]
protected_hashes={str(p.relative_to(DATA)):hashlib.sha256(p.read_bytes()).hexdigest() for p in protected_paths}
print("Positive Funding/Other weights only; source DBSCAN labels, C=1000, scaler, and other classifiers frozen.")


Positive Funding/Other weights only; source DBSCAN labels, C=1000, scaler, and other classifiers frozen.


In [2]:
def predict_model(model,raw):
    scaled=model["scaler"].transform(np.asarray(raw))
    return np.column_stack([e.predict_proba(scaled)[:,list(e.classes_).index(1)] for e in model["estimators"]])


def label_predictions(probabilities):
    assert np.isfinite(probabilities).all() and ((probabilities>=0)&(probabilities<=1)).all()
    predicted=(probabilities>=0.5).astype(int);empty=predicted.sum(axis=1)==0
    rows=np.flatnonzero(empty);predicted[rows,probabilities[rows].argmax(axis=1)]=1
    return predicted,empty


def effective_weights(reason,multiplier):
    j=REASONS.index(reason);weights=W[:,j].copy();weights[Y[:,j]==1]*=multiplier
    weights*=157/weights.sum()
    assert np.array_equal(weights>0,W[:,j]>0)
    np.testing.assert_allclose(weights.sum(),157,atol=1e-12)
    return weights

reference_probability=predict_model(source_model,valid[FEATURES].to_numpy())
reference_prediction,_=label_predictions(reference_probability)
reference_sample=f1_score(truth,reference_prediction,average="samples",zero_division=0)
reference_macro=f1_score(truth,reference_prediction,average="macro",zero_division=0)
np.testing.assert_allclose(reference_sample,record["validation_sample_f1"],atol=1e-14)
CACHE={};WEIGHT_AUDIT=[]
with threadpool_limits(limits=1):
    for reason in TARGET_REASONS:
        j=REASONS.index(reason)
        for multiplier in POSITIVE_MULTIPLIERS:
            weights=effective_weights(reason,multiplier);included=weights>0
            estimator=clone(source_model["estimators"][j])
            assert estimator.C==1000 and estimator.class_weight is None
            with warnings.catch_warnings():
                warnings.simplefilter("error",ConvergenceWarning)
                estimator.fit(XS[included],Y[included,j],sample_weight=weights[included])
            if multiplier==1:
                np.testing.assert_allclose(estimator.predict_proba(VS),source_model["estimators"][j].predict_proba(VS),rtol=1e-8,atol=1e-10)
            CACHE[(reason,multiplier)]=estimator
            positive=(Y[:,j]==1)&included;negative=(Y[:,j]==0)&included
            WEIGHT_AUDIT.append({"reason":reason,"positive_multiplier":multiplier,"fitting_rows":int(included.sum()),
                                 "positive_rows":int(positive.sum()),"negative_rows":int(negative.sum()),
                                 "effective_positive_weight_total":float(weights[positive].sum()),
                                 "effective_negative_weight_total":float(weights[negative].sum()),
                                 "effective_weight_total":float(weights.sum())})
pd.DataFrame(WEIGHT_AUDIT).to_csv(OUT/"weight_audit.csv",index=False)
print("Eight reason-specific fits completed; 1x weights reproduce the original Funding and Other models.")


Eight reason-specific fits completed; 1x weights reproduce the original Funding and Other models.


In [3]:
ROWS=[]
for funding_weight in POSITIVE_MULTIPLIERS:
    for other_weight in POSITIVE_MULTIPLIERS:
        name=f"funding{funding_weight:g}_other{other_weight:g}".replace(".","p")
        multipliers={"Funding":funding_weight,"Other":other_weight}
        estimators=list(source_model["estimators"])
        for reason,multiplier in multipliers.items():estimators[REASONS.index(reason)]=CACHE[(reason,multiplier)]
        model={**source_model,"estimators":estimators,"positive_class_multipliers":multipliers,
               "source_dataset":"datasets/fixed_source.csv","effective_weight_file":f"datasets/{name}_effective_weights.csv",
               "prediction_cutoff":0.5,"nonempty_fallback":True}
        weights_frame=train[["ID"]].copy()
        for reason in REASONS:
            j=REASONS.index(reason);weights=effective_weights(reason,multipliers.get(reason,1.0))
            weights_frame[f"effective_weight_{reason}"]=weights
            if reason not in TARGET_REASONS:
                np.testing.assert_array_equal(model["estimators"][j].coef_,source_model["estimators"][j].coef_)
                np.testing.assert_array_equal(model["estimators"][j].intercept_,source_model["estimators"][j].intercept_)
        weights_frame.to_csv(OUT/"datasets"/f"{name}_effective_weights.csv",index=False)
        probabilities=predict_model(model,valid[FEATURES].to_numpy())
        predicted,empty=label_predictions(probabilities)
        sample=f1_score(truth,predicted,average="samples",zero_division=0)
        macro=f1_score(truth,predicted,average="macro",zero_division=0)
        per_reason=f1_score(truth,predicted,average=None,zero_division=0)
        counts={}
        for reason in TARGET_REASONS:
            j=REASONS.index(reason);actual=truth[:,j];p=predicted[:,j]
            tp=int(((actual==1)&(p==1)).sum());fp=int(((actual==0)&(p==1)).sum());fn=int(((actual==1)&(p==0)).sum())
            counts.update({f"tp_{reason}":tp,f"fp_{reason}":fp,f"fn_{reason}":fn,
                           f"precision_{reason}":tp/(tp+fp) if tp+fp else 0.0,f"recall_{reason}":tp/(tp+fn) if tp+fn else 0.0})
        ROWS.append({"experiment":name,"funding_positive_weight":funding_weight,"other_positive_weight":other_weight,
                     "sample_f1":sample,"macro_f1":macro,**dict(zip([f"f1_{r}" for r in REASONS],per_reason)),**counts,
                     "fallback_trials":int(empty.sum()),"changed_trials_vs_current":int((predicted!=reference_prediction).any(axis=1).sum()),
                     "delta_vs_current":sample-reference_sample,"is_reference":funding_weight==1 and other_weight==1,
                     "distance_from_reference":float(np.log(funding_weight)+np.log(other_weight)),"kaggle_score":np.nan})
        frame=valid[["ID"]+REASONS].rename(columns={r:f"actual_{r}" for r in REASONS}).copy()
        for j,r in enumerate(REASONS):frame[f"probability_{r}"]=probabilities[:,j];frame[f"predicted_{r}"]=predicted[:,j]
        frame["used_fallback"]=empty
        frame["trial_f1"]=2*(truth*predicted).sum(axis=1)/(truth.sum(axis=1)+predicted.sum(axis=1))
        frame.to_csv(OUT/"validation_predictions"/f"{name}.csv",index=False)
        joblib.dump(model,OUT/"models"/f"{name}.joblib",compress=3)
comparison=pd.DataFrame(ROWS);assert len(comparison)==16
comparison.to_csv(OUT/"comparison.csv",index=False)
ranked=comparison.sort_values(["sample_f1","is_reference","distance_from_reference","funding_positive_weight","other_positive_weight"],
                             ascending=[False,False,True,True,True],kind="stable")
best=ranked.iloc[0]
print("Sample F1 by multiplier:")
display(comparison.pivot(index="funding_positive_weight",columns="other_positive_weight",values="sample_f1").round(6))
print("Selected:",best.experiment,"sample F1:",best.sample_f1,"macro F1:",best.macro_f1)
display(ranked.head(8)[["funding_positive_weight","other_positive_weight","sample_f1","macro_f1","f1_Funding","f1_Other","delta_vs_current"]].round(6))


Sample F1 by multiplier:


other_positive_weight,1.0,1.5,2.0,3.0
funding_positive_weight,,,,
1.0,0.910256,0.910256,0.910256,0.910256
1.5,0.910256,0.910256,0.910256,0.910256
2.0,0.910256,0.910256,0.910256,0.910256
3.0,0.910256,0.910256,0.910256,0.910256


Selected: funding1_other1 sample F1: 0.9102564102564102 macro F1: 0.811421460284173


,funding_positive_weight,other_positive_weight,sample_f1,macro_f1,f1_Funding,f1_Other,delta_vs_current
0,1.0,1.0,0.910256,0.811421,0.666667,0.631579,0.0
1,1.0,1.5,0.910256,0.811421,0.666667,0.631579,0.0
4,1.5,1.0,0.910256,0.811421,0.666667,0.631579,0.0
2,1.0,2.0,0.910256,0.811421,0.666667,0.631579,0.0
8,2.0,1.0,0.910256,0.811421,0.666667,0.631579,0.0
5,1.5,1.5,0.910256,0.811421,0.666667,0.631579,0.0
6,1.5,2.0,0.910256,0.811421,0.666667,0.631579,0.0
9,2.0,1.5,0.910256,0.811421,0.666667,0.631579,0.0


In [4]:
selected=joblib.load(OUT/"models"/f"{best.experiment}.joblib")
selected_probability=predict_model(selected,valid[FEATURES].to_numpy())
for row in comparison.itertuples(index=False):
    model=joblib.load(OUT/"models"/f"{row.experiment}.joblib")
    frame=pd.read_csv(OUT/"validation_predictions"/f"{row.experiment}.csv",float_precision="round_trip")
    assert frame.ID.tolist()==valid.ID.tolist()
    probs=frame[[f"probability_{r}" for r in REASONS]].to_numpy();prediction=frame[[f"predicted_{r}" for r in REASONS]].to_numpy(dtype=int)
    np.testing.assert_allclose(predict_model(model,valid[FEATURES].to_numpy()),probs,rtol=1e-12,atol=1e-12)
    np.testing.assert_array_equal(label_predictions(probs)[0],prediction)
    np.testing.assert_allclose(f1_score(truth,prediction,average="samples",zero_division=0),row.sample_f1,atol=1e-14)
    np.testing.assert_allclose(f1_score(truth,prediction,average="macro",zero_division=0),row.macro_f1,atol=1e-14)
    effective=pd.read_csv(OUT/"datasets"/f"{row.experiment}_effective_weights.csv",float_precision="round_trip")
    assert effective.ID.tolist()==train.ID.tolist()
    for reason in REASONS:
        weights=effective[f"effective_weight_{reason}"].to_numpy()
        np.testing.assert_allclose(weights,effective_weights(reason,model["positive_class_multipliers"].get(reason,1.0)),rtol=1e-12,atol=1e-12)
# Independently refit all five selected estimators from persisted shared data/effective weights.
frame=pd.read_csv(OUT/"datasets/fixed_source.csv",float_precision="round_trip")
assert frame.ID.tolist()==train.ID.tolist()
np.testing.assert_array_equal(frame[REASONS].to_numpy(dtype=int),Y)
np.testing.assert_array_equal(frame[[f"weight_{r}" for r in REASONS]].to_numpy(),W)
effective=pd.read_csv(OUT/selected["effective_weight_file"],float_precision="round_trip")
scaled=scaler.transform(frame[FEATURES].to_numpy());replay={**selected,"estimators":[]}
with threadpool_limits(limits=1):
    for j,reason in enumerate(REASONS):
        weights=effective[f"effective_weight_{reason}"].to_numpy();included=weights>0
        estimator=clone(selected["estimators"][j])
        with warnings.catch_warnings():
            warnings.simplefilter("error",ConvergenceWarning)
            estimator.fit(scaled[included],Y[included,j],sample_weight=weights[included])
        replay["estimators"].append(estimator)
np.testing.assert_allclose(predict_model(replay,valid[FEATURES].to_numpy()),selected_probability,rtol=1e-8,atol=1e-10)
np.testing.assert_array_equal(label_predictions(predict_model(replay,valid[FEATURES].to_numpy()))[0],label_predictions(selected_probability)[0])
for rel,h in protected_hashes.items():assert hashlib.sha256((DATA/rel).read_bytes()).hexdigest()==h
print("All 16 saved models/scores/weight matrices, original control, unchanged classifiers, independent selected-model refit, and protected hashes passed.")


All 16 saved models/scores/weight matrices, original control, unchanged classifiers, independent selected-model refit, and protected hashes passed.


In [5]:
candidate=None
if best.sample_f1>reference_sample+1e-12:
    test=pd.read_csv(INPUT_DIR/"part1_test.csv");sample=pd.read_csv(INPUT_DIR/"part1_sample_submission.csv")
    assert test.ID.is_unique and sample.ID.is_unique and set(test.ID)==set(sample.ID)
    assert set(test.ID).isdisjoint(train.ID) and set(test.ID).isdisjoint(valid.ID)
    prediction,empty=label_predictions(predict_model(selected,test[FEATURES].to_numpy()))
    output=pd.DataFrame({"ID":test.ID,"Predicted":[" ".join(r for r,yes in zip(REASONS,row) if yes) for row in prediction]})
    output=output.set_index("ID").loc[sample.ID].reset_index()
    assert output.shape==(312,2) and output.Predicted.str.len().gt(0).all()
    path=OUT/"submissions"/f"part1_dbscan_{best.experiment}_submission.csv"
    output.to_csv(path,index=False);pd.testing.assert_frame_equal(pd.read_csv(path),output)
    candidate={"experiment":best.experiment,"validation_sample_f1":float(best.sample_f1),"validation_macro_f1":float(best.macro_f1),
               "submission_file":str(path.relative_to(OUT)),"submission_sha256":hashlib.sha256(path.read_bytes()).hexdigest(),
               "model_file":f"models/{best.experiment}.joblib","model_sha256":hashlib.sha256((OUT/"models"/f"{best.experiment}.joblib").read_bytes()).hexdigest(),
               "test_rows":312,"test_fallback_trials":int(empty.sum()),"kaggle_score":None,"kaggle_status":"prepared locally; not submitted"}
    (OUT/"submission_manifest.json").write_text(json.dumps(candidate,indent=2)+"\n")
settings={"positive_multipliers":POSITIVE_MULTIPLIERS,"target_reasons":TARGET_REASONS,"configurations":16,
          "reference_sample_f1":reference_sample,"reference_macro_f1":reference_macro,
          "source_targets_changed":False,"source_raw_weights_changed":False,"generated_labels_changed":False,
          "effective_fit_weight_method":"multiply raw positive Funding/Other weights, then normalize each reason to total 157",
          "other_classifiers_refitted":False,"classifier_C":1000,"features":FEATURES,"classifier_scaler_fit_rows":157,
          "cutoff":0.5,"fallback":True,"validation_rows":78,"validation_used_for_fit":False,"final_validation_label_refit":False,
          "best_experiment":best.experiment,"best_sample_f1":float(best.sample_f1),"best_macro_f1":float(best.macro_f1),
          "best_funding_multiplier":float(best.funding_positive_weight),"best_other_multiplier":float(best.other_positive_weight),
          "leader_changed":bool(best.sample_f1>reference_sample+1e-12),"candidate":candidate,
          "protected_input_sha256":protected_hashes,
          "dataset_file_sha256":{str(p.relative_to(OUT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted((OUT/"datasets").glob('*.csv'))},
          "selection":"sample F1; ties prefer original control, then smaller positive multipliers",
          "versions":{"python":platform.python_version(),"sklearn":sklearn.__version__,"numpy":np.__version__,"pandas":pd.__version__},"new_kaggle_scores":None}
(OUT/"settings.json").write_text(json.dumps(settings,indent=2)+"\n")
lines=["# Funding and Other positive-class weighting","",
       "16 classifier settings cross positive multipliers 1/1.5/2/3 for Funding and Other, including the original 1/1 control. "
       "Source DBSCAN targets/raw weights stay unchanged. Only effective fitting weights for those two binary classifiers "
       "change: multiply positive weights, leave negative weights unchanged, then normalize each reason's total to 157. "
       "Safety, Efficacy, and Recruitment estimators are reused unchanged. All classifiers retain C=1000, original 150 "
       "embeddings/scaler, cutoff 0.5, and highest-probability fallback. No new class priors are assumed to be population rates.","",
       f"Reference sample F1 **{reference_sample:.6f}**, macro F1 **{reference_macro:.6f}**. "
       f"Selected **{best.experiment}**: sample F1 **{best.sample_f1:.6f}**, macro F1 **{best.macro_f1:.6f}**.","",
       "| Funding positive weight | Other positive weight | Sample F1 | Macro F1 | Funding F1 | Other F1 | Change |",
       "|---:|---:|---:|---:|---:|---:|---:|"]
for row in comparison.itertuples(index=False):
    lines.append(f"| {row.funding_positive_weight:g} | {row.other_positive_weight:g} | {row.sample_f1:.6f} | {row.macro_f1:.6f} | {row.f1_Funding:.6f} | {row.f1_Other:.6f} | {row.delta_vs_current:+.6f} |")
lines.extend(["","The original control reproduces its probabilities. All saved models and prediction sets reload and "
       "rescore correctly; saved effective weights reproduce the declared multipliers and fixed total. The selected "
       "model independently refits from its saved source data and effective weights with matching probabilities. "
       "The untouched three classifiers and protected source files retain their parameters/checksums.","",
       "datasets/ saves the shared original data plus exact effective weights per configuration; models/ and "
       "validation_predictions/ preserve all fits and outputs. comparison.csv includes per-reason F1 and Funding/Other "
       "precision, recall, true positives, false positives, and false negatives; weight_audit.csv records effective class "
       "weight totals; settings.json preserves settings and hashes.","",
       "This last bounded screen uses the same repeatedly consulted 78-trial validation set and does not establish "
       "independent test improvement. No validation labels enter fitting, and no final refit, Kaggle submission, "
       "commit, or tag was performed.",""])
lines.append(f"New candidate prepared locally: `{candidate['submission_file']}`. Kaggle score unknown." if candidate else
             "No setting improves primary sample F1; keep the existing DBSCAN leader and its prepared Kaggle file.")
(OUT/"SUMMARY.md").write_text("\n".join(lines)+"\n")
print("Best:",best.experiment,"sample F1:",best.sample_f1,"macro F1:",best.macro_f1,"Leader changed:",settings['leader_changed'])
print("Candidate:",candidate)


Best: funding1_other1 sample F1: 0.9102564102564102 macro F1: 0.811421460284173 Leader changed: False
Candidate: None


## Weight-effect diagnostic

All 16 settings produce exactly the same validation label sets as the original model. The weights are active: at Funding=3 and Other=3, fitted coefficients change and the maximum absolute validation probability changes are approximately **0.02443 for Funding** and **0.35611 for Other**. None of these changes alters the final cutoff/fallback decisions. Safety, Efficacy, and Recruitment coefficients and probabilities remain exactly unchanged. `weight_effect_diagnostic.json` records these checks. Retain the original model and its existing prepared Kaggle submission.